<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-08/AI_Modul_8.1_Praktikum_Preprocessing_Dataset.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 8.1: Praktikum Preprocessing Dataset
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Memahami prosedur penyiapan data multivariat agrokompleks (spektrometri Vis-NIR daun dan sensor IoT perkebunan) menjadi representasi tensor.
2. Membuktikan secara empiris dampak negatif kebocoran data spasial (*spatial data leakage*) dan mengimplementasikan *Spatial Block Partitioning*.
3. Membangun modul standarisasi statistik terisolasi (*zero data leakage scaler*).
4. Mengonstruksi kelas kustom `torch.utils.data.Dataset` dan mengonfigurasi generator batch `torch.utils.data.DataLoader` berkinerja tinggi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader

# Konfigurasi seed komputasi untuk reproduktibilitas eksperimen
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print(f"NumPy Version  : {np.__version__}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")


## 1. Pembangkitan Dataset Agrokompleks Multivariat
Kita merekayasa data sintetis terstruktur yang merefleksikan 600 pohon kelapa sawit dari 10 Afdeling perkebunan:
- 120 pita spektral Vis-NIR (400 - 1000 nm).
- 4 parameter sensor IoT (pH tanah, Nitrogen tanah, kelembaban, curah hujan).
- 1 target regresi kontinu: Rendemen CPO (%).


In [ ]:
n_samples = 600
n_spectral = 120

# 10 Afdeling perkebunan (60 pohon per afdeling)
afdeling_ids = np.repeat(np.arange(1, 11), n_samples // 10)

# Koordinat spasial sintetis untuk visualisasi blok perkebunan
afdeling_x = (afdeling_ids - 1) % 5 * 2.0 + np.random.normal(0, 0.3, n_samples)
afdeling_y = (afdeling_ids - 1) // 5 * 2.0 + np.random.normal(0, 0.3, n_samples)

# Fitur Spektral Vis-NIR
wavelengths = np.linspace(400, 1000, n_spectral)
X_spectral = np.zeros((n_samples, n_spectral))
for i in range(n_samples):
    base_sig = 0.15 + 0.35 * np.exp(-((wavelengths - 550)**2) / (2 * 45**2)) +                       0.50 * (1.0 / (1.0 + np.exp(-(wavelengths - 700) / 20)))
    noise = np.random.normal(0, 0.02, n_spectral)
    X_spectral[i] = base_sig * np.random.uniform(0.85, 1.15) + noise

# Fitur Sensor Tanah & Lingkungan
pH = np.random.normal(5.2, 0.4, (n_samples, 1))
nitrogen = np.random.normal(120.0, 25.0, (n_samples, 1))
moisture = np.random.uniform(30.0, 70.0, (n_samples, 1))
rainfall = np.random.uniform(100.0, 350.0, (n_samples, 1))
X_sensor = np.hstack([pH, nitrogen, moisture, rainfall])

# Penggabungan seluruh fitur
X_raw = np.hstack([X_spectral, X_sensor])

# Target Rendemen CPO (%)
y_raw = 19.5 + 4.5 * X_spectral[:, 65] - 0.015 * np.abs(pH[:, 0] - 5.5)**2 +         0.012 * (nitrogen[:, 0] / 10.0) + np.random.normal(0, 0.35, n_samples)
y_raw = y_raw.reshape(-1, 1)

print(f"Total Sampel Data : {X_raw.shape[0]} observasi pohon")
print(f"Dimensi Fitur     : {X_raw.shape[1]} kolom (120 spektral + 4 sensor tanah)")
print(f"Dimensi Target    : {y_raw.shape} (Rendemen Minyak CPO %)")


## 2. Partisi Berbasis Blok Spasial (*Spatial Block Partitioning*)
Kita memisahkan dataset berdasarkan unit afdeling geografis perkebunan:
- **Data Latih (Train):** Afdeling 1 s.d. 6 (60% blok kebun).
- **Data Validasi (Val):** Afdeling 7 & 8 (20% blok kebun).
- **Data Uji (Test):** Afdeling 9 & 10 (20% blok kebun).


In [ ]:
train_blocks = [1, 2, 3, 4, 5, 6]
val_blocks   = [7, 8]
test_blocks  = [9, 10]

train_mask = np.isin(afdeling_ids, train_blocks)
val_mask   = np.isin(afdeling_ids, val_blocks)
test_mask  = np.isin(afdeling_ids, test_blocks)

X_train, y_train = X_raw[train_mask], y_raw[train_mask]
X_val, y_val     = X_raw[val_mask], y_raw[val_mask]
X_test, y_test   = X_raw[test_mask], y_raw[test_mask]

print(f"Sampel Latih   : {X_train.shape[0]} pohon (Afdeling {train_blocks})")
print(f"Sampel Validasi: {X_val.shape[0]} pohon (Afdeling {val_blocks})")
print(f"Sampel Uji     : {X_test.shape[0]} pohon (Afdeling {test_blocks})")


## 3. Visualisasi Partisi Spasial dan Profil Spektrometri Daun


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sebaran Spasial Pohon Perkebunan
axes[0].scatter(afdeling_x[train_mask], afdeling_y[train_mask], c='forestgreen', alpha=0.7, s=40, label='Train Set (Afdeling 1-6)')
axes[0].scatter(afdeling_x[val_mask], afdeling_y[val_mask], c='darkorange', alpha=0.8, s=45, label='Val Set (Afdeling 7-8)')
axes[0].scatter(afdeling_x[test_mask], afdeling_y[test_mask], c='crimson', alpha=0.9, s=55, marker='^', label='Test Set (Afdeling 9-10)')
axes[0].set_title('Partisi Spasial Blok Perkebunan Bebas Leakage', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Koordinat Sumbu X Afdeling')
axes[0].set_ylabel('Koordinat Sumbu Y Afdeling')
axes[0].legend(loc='lower right')

# Plot 2: Profil Reflektansi Spektral Daun Sawit
for i in range(10):
    axes[1].plot(wavelengths, X_spectral[i], alpha=0.6, lw=1.2)
axes[1].set_title('Profil Spektrometri Vis-NIR Daun Kelapa Sawit (10 Sampel)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Panjang Gelombang (nm)')
axes[1].set_ylabel('Faktor Reflektansi')

plt.tight_layout()
plt.show()


## 4. Implementasi Scaler Terisolasi (*Zero Data Leakage*)
Kita menghitung parameter statistik $\boldsymbol{\mu}_{train}$ dan $\boldsymbol{\sigma}_{train}$ hanya dari subset data latih, lalu mentransformasikannya ke data validasi dan data uji.


In [ ]:
class IsolatedStandardScaler:
    """Scaler statistik terisolasi untuk mencegah kebocoran data validasi/uji."""
    def __init__(self, eps=1e-8):
        self.eps = eps
        self.mu = None
        self.sigma = None

    def fit(self, X):
        self.mu = np.mean(X, axis=0, keepdims=True)
        self.sigma = np.std(X, axis=0, keepdims=True)
        self.sigma[self.sigma == 0] = 1.0  # cegah pembagian nol
        return self

    def transform(self, X):
        if self.mu is None or self.sigma is None:
            raise RuntimeError("Scaler wajib di-fit terlebih dahulu sebelum mentransformasi data!")
        return (X - self.mu) / (self.sigma + self.eps)

    def fit_transform(self, X):
        return self.fit(X).transform(X)

# Eksekusi standarisasi
scaler_X = IsolatedStandardScaler()
X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm   = scaler_X.transform(X_val)
X_test_norm  = scaler_X.transform(X_test)

print("Verifikasi Nilai Statistik Data Latih Terstandarisasi:")
print(f"Rata-rata fitur (harus mendekati 0.0): {np.mean(X_train_norm):.6f}")
print(f"Standar deviasi (harus mendekati 1.0) : {np.std(X_train_norm):.6f}")

print("")
print("Verifikasi Nilai Statistik Data Validasi (Wajar sedikit bergeser dari 0):")
print(f"Rata-rata fitur data validasi        : {np.mean(X_val_norm):.4f}")
print(f"Standar deviasi data validasi         : {np.std(X_val_norm):.4f}")


## 5. Konstruksi PyTorch Custom Dataset & DataLoader
Kita membungkus tensor masukan dan target ke dalam kelas turunan `torch.utils.data.Dataset` dan mengonfigurasi `DataLoader` berkinerja tinggi.


In [ ]:
class OilPalmSpectralDataset(Dataset):
    """Dataset PyTorch kustom untuk data spektral dan sensor IoT agrokompleks."""
    def __init__(self, features, targets):
        # Konversi ke tensor float32
        self.features = torch.tensor(features, dtype=torch.float32)
        self.targets  = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.targets[idx]

# Instansiasi objek Dataset
train_dataset = OilPalmSpectralDataset(X_train_norm, y_train)
val_dataset   = OilPalmSpectralDataset(X_val_norm, y_val)
test_dataset  = OilPalmSpectralDataset(X_test_norm, y_test)

# Konfigurasi DataLoader mini-batch
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Ukuran batch        : {batch_size} sampel per iterasi")
print(f"Jumlah batch Train  : {len(train_loader)} batch per epoch")
print(f"Jumlah batch Val    : {len(val_loader)} batch per epoch")
print(f"Jumlah batch Test   : {len(test_loader)} batch per epoch")


## 6. Uji Aliran Tensor & Verifikasi Dimensi Batch
Kita menginspeksi dimensi dan tipe data tensor yang dihasilkan oleh iterator `DataLoader` pada batch pertama.


In [ ]:
# Ambil satu batch dari train_loader
first_batch_x, first_batch_y = next(iter(train_loader))

print("=================================================================")
print("             INSPEKSI STRUKTUR TENSOR MINI-BATCH                 ")
print("=================================================================")
print(f"Bentuk Tensor Fitur (batch_x) : {first_batch_x.shape}")
print(f"Tipe Data Fitur               : {first_batch_x.dtype}")
print(f"Bentuk Tensor Target (batch_y): {first_batch_y.shape}")
print(f"Tipe Data Target              : {first_batch_y.dtype}")
print(f"Rentang Nilai Fitur           : [{first_batch_x.min().item():.3f}, {first_batch_x.max().item():.3f}]")
print(f"Rentang Nilai Target          : [{first_batch_y.min().item():.3f}, {first_batch_y.max().item():.3f}] %")
print("=================================================================")
print("[OK] Pipeline Preprocessing dan DataLoader siap diintegrasikan ke Training Pipeline!")
